# Lab E — AMD CPU + GPU Heterogeneous Pipeline / CPU+GPU 异步流水线
## Slide 49–50 · 让 CPU 和 GPU 同时干活

**What will you learn? / 本节学什么？**
- 串行处理的等待在哪里 Where serial processing wastes time waiting
- CPU 预处理与 GPU 推理重叠的异步流水线如何工作 How an async pipeline overlaps CPU preprocessing with GPU inference
- 为什么吞吐可能提高、单张延迟却未必下降 Why throughput can rise while per-image latency may not

**One key idea / 一个核心原理**
> 串行时 GPU 在算第 i 张图、CPU 在空闲；流水线让 CPU 趁 GPU 计算时准备第 i+1 张图——等待被“藏”起来了。Serially the CPU idles while the GPU computes; > pipelining hides that wait by preparing image i+1 during GPU compute of i.

In [ ]:
from pathlib import Path
import sys
REPO = Path.cwd()
if REPO.name == "notebooks":
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
print("repo:", REPO)

## 1. The idea / 原理
两条路径使用**完全相同**的检测逻辑（读取 → letterbox 预处理 → YOLO26l 前向 → 后处理），处理**同一批图片**：
- **串行 Serial**：一张图完整走完全部步骤，再处理下一张。
- **异步流水线 Async pipeline**：CPU 线程准备第 i+1 张图的同时，GPU 线程计算第 i 张图（有界队列实现双缓冲）。

Both paths share ONE detection pipeline (load → letterbox → forward → post) on the same images. Serial runs stages one-at-a-time; the async pipeline overlaps CPU prepare of image i+1 with GPU compute of image i (bounded queue = double buffering).

In [ ]:
from course.benchmark import load_bench_images
from course.pipeline import run_serial, run_pipeline, compare_results

STUDENT_N_IMAGES = 60                 # <-- student-adjustable (up to 120)
STUDENT_QUEUE_DEPTH = 4                # <-- student-adjustable double-buffer depth

images = load_bench_images(STUDENT_N_IMAGES)
print(f"{len(images)} images ready (official YOLO26l on W7900 for both modes)")

## 2. Run the baseline: serial / 基线：串行处理

In [ ]:
serial = run_serial(images, imgsz=640)
print(f"serial   : total {serial['total_ms']/1000:.1f} s · "
      f"{serial['images_per_s']:.1f} img/s · e2e P50 {serial['e2e_p50_ms']:.0f} ms")

## 3. Change one thing: async CPU+GPU pipeline / 只改一件事：异步流水线
线程、队列与同步都封装在课程工具模块里，我们只调 `run_pipeline()`。
Threads/queues/sync are hidden in the course module — we just call `run_pipeline()`.

In [ ]:
pipeline = run_pipeline(images, imgsz=640, queue_depth=STUDENT_QUEUE_DEPTH)
print(f"pipeline : total {pipeline['total_ms']/1000:.1f} s · "
      f"{pipeline['images_per_s']:.1f} img/s · e2e P50 {pipeline['e2e_p50_ms']:.0f} ms")

## 4. Compare the results / 对比结果
除了速度，还要验证两条路径的**检测结果一致**（同样的图、同样的逻辑 → 同样的框）。
Besides speed, verify both paths produce **identical detections** (same images + same logic → same boxes).

In [ ]:
import pandas as pd
cmp = compare_results(serial, pipeline)
display(pd.DataFrame([
    {"mode": "Serial 串行", "total (s)": round(serial['total_ms']/1000, 1),
     "img/s": round(serial['images_per_s'], 1),
     "e2e P50 (ms)": round(serial['e2e_p50_ms'], 0),
     "detections consistent": cmp['consistent']},
    {"mode": "Async pipeline 流水线", "total (s)": round(pipeline['total_ms']/1000, 1),
     "img/s": round(pipeline['images_per_s'], 1),
     "e2e P50 (ms)": round(pipeline['e2e_p50_ms'], 0),
     "detections consistent": cmp['consistent']}]))
print(f"throughput gain: {cmp['throughput_gain_x']:.2f}x · "
      f"mismatched images: {len(cmp['detection_mismatches'])}")

In [ ]:
from course.charts import chart_pipeline
chart_pipeline(serial, pipeline)

## 5. Why? / 为什么会这样？
- 吞吐提高：GPU 不再等 CPU 读图/预处理，总时间 ≈ max(CPU 阶段, GPU 阶段) × N，而不是两者相加。
- 单张延迟未必下降：每张图仍要走完所有阶段，甚至多了入队/唤醒开销；如果 CPU 预处理比 GPU 推理慢，瓶颈就转移到 CPU——收益由**端到端实测**决定。

- Throughput rises because the GPU no longer waits for CPU work: total time ≈ max(CPU stage, GPU stage) × N instead of their sum.
- Per-image latency may NOT drop: every image still passes all stages (plus queueing). If CPU preprocessing is slower than GPU inference, the bottleneck moves to the CPU. Gains are decided by end-to-end measurement.

## 6. Try it yourself / 学生练习
把 `QUEUE_DEPTH` to 1 or 8 (or `N_IMAGES` 改成 `120`，重新运行，解释吞吐和延迟的变化。瓶颈在 CPU 还是 GPU？（提示：看上一步的 `pre_p50` 与 `infer_p50`。）
Change `STUDENT_QUEUE_DEPTH` to 1 or 8 (or `STUDENT_N_IMAGES` to 120), re-run and explain. Where is the bottleneck? (Hint: compare `pre_p50` vs `infer_p50`.)

**Takeaway / 一句话总结**
> 让 CPU 与 GPU 合理协同可以隐藏等待、提高吞吐——但收益大小取决于瓶颈在哪，必须端到端实测。
> Overlapping CPU and GPU work hides waiting and can boost throughput — > but the gain depends on where the bottleneck is, and must be proven > end-to-end.